<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/LLaDA_GSM8K_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/LLaDA_GSM8K_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LLaDA on GSM8K

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


GPU: NVIDIA L4 | GPU memory: 22.0 GiB


## 1. Get the code from `adi`
Clone or update `adi`. A clean checkout on another branch can switch safely; local modifications and divergent commits are never discarded.


In [2]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

ensure_checkout()


Code ready: /content/CSCI544-DLM | branch: adi | commit: c63ceb0


## 2. Install LLaDA dependencies
Keep Colab's CUDA-enabled PyTorch. Inference always uses a fresh Python process, which sees the installed dependency versions. LLaDA retains Transformers 4.38.2.

If you switch model notebooks in the same runtime, run this installation cell again before inference. The two model profiles use different Transformers versions.


In [3]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_DIR / "requirements/llada-colab.txt"),
], check=True)

# Install our package without replacing Colab's CUDA-enabled PyTorch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
# Refresh editable-package paths in this already-running notebook kernel.
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)


## 3. Mount Drive
Allow at least **20 GB free Drive storage per model snapshot**, plus dataset cache and experiment outputs. Downloading both models needs room for both copies. Colab Pro compute and Drive storage are separate allowances.

`COPY_MODEL_TO_RUNTIME = True` stages the persistent model on runtime disk. Set it to `False` to load directly from Drive. Existing experiment folders are not moved or deleted.


In [4]:
from google.colab import drive
import importlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
MODEL_FAMILY = "llada"
from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
profile = model_profiles.get_profile(MODEL_FAMILY)
MODEL_NAME = profile["repo_id"].split("/")[-1]
DRIVE_MODEL_DIR = DRIVE_ROOT / "models" / MODEL_NAME
RUNTIME_MODEL_DIR = Path("/content/models") / MODEL_NAME
COPY_MODEL_TO_RUNTIME = True
EXPERIMENT_TIMEZONE = "America/Los_Angeles"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
MODEL_PATH = RUNTIME_MODEL_DIR if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIR
print("Persistent model:", DRIVE_MODEL_DIR)
print("Experiment root:", DRIVE_ROOT / profile["label"] / "GSM8K")


Mounted at /content/drive
Persistent model: /content/drive/MyDrive/CSCI544-DLM/models/LLaDA-8B-Instruct
Experiment root: /content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K


## 4. Load the GSM8K configuration
Use the same model/cache locations and timezone as Sudoku. Outputs go under `LLaDA/GSM8K/`, keeping this task's results together. Edit the question and instructions in Config Updater below.


In [5]:
import yaml

config = yaml.safe_load((REPO_DIR / "configs/gsm8k/llada.yaml").read_text())
config["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
config["model"].update(family=MODEL_FAMILY, model_path=str(MODEL_PATH), device="cuda", local_files_only=True)
config["experiment_root"] = str(DRIVE_ROOT)
config["experiment_timezone"] = EXPERIMENT_TIMEZONE

# Update the config to give some few shot examples
config["prompt"]["instructions"] = """Solve the next math problem in English.
Follow the examples: use brief calculations and finish with a separate
line containing #### followed by a plain number.
Do not add units, %, LaTeX, or text after the final number.

Question:
Weng earns $12 an hour for babysitting. Yesterday, she babysat for
5 hours. She spent $15 on lunch. How much money does she have left?
Answer:
12 * 5 = 60. After lunch, 60 - 15 = 45.
#### 45

Question:
Shawn has five toys. For Christmas, he got two toys each from his
mom and dad. How many toys does he have now?
Answer:
He receives 2 + 2 = 4 toys. His total is 5 + 4 = 9.
#### 9

Question:
Jason had 20 lollipops. He gave 4 to Denny and 5 to Michael.
How many lollipops does Jason have left?
Answer:
He gives away 4 + 5 = 9. He has 20 - 9 = 11 left.
#### 11

Question:
Bella bought two packs of gum with 10 pieces each. She chewed
4 pieces on Monday and 3 pieces on Tuesday. How many pieces are left?
Answer:
She bought 2 * 10 = 20 pieces and chewed 4 + 3 = 7.
There are 20 - 7 = 13 left.
#### 13

Now solve the following question using the same answer format."""


RUN_CONFIG = Path("/content/llada-gsm8k-run.yaml")
RUNNER = REPO_DIR / "scripts/run_gsm8k.py"
print("Model revision:", profile["revision"])


Model revision: 08b83a6feb34df1a6011b80c3c00c7563e963b07


## 5. Cache the model on Drive
Download the pinned model's tokenizer, chat template, configuration, Python implementation, and all shards listed in its weight index. Later sessions reuse the same files; keep the model folder's hidden `.cache` directory.

Rerun this cell after an interrupted download. If validation reports missing files or insufficient disk space, resolve that before inference. No weights are committed to GitHub.


In [6]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", MODEL_FAMILY, "--drive-dir", str(DRIVE_MODEL_DIR)]
if COPY_MODEL_TO_RUNTIME:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/CSCI544-DLM/scripts/cache_model.py', '--model', 'llada', '--drive-dir', '/content/drive/MyDrive/CSCI544-DLM/models/LLaDA-8B-Instruct', '--runtime-dir', '/content/models/LLaDA-8B-Instruct'], returncode=0)

## Benchmark and dataset Configs

In [8]:
from pathlib import Path
import os
import yaml

BENCH_CONFIG = Path("/content/gsm8k-decoding.yaml")

cfg = yaml.safe_load(
    (REPO_DIR / "configs/gsm8k/decoding.yaml").read_text()
)
cfg["model"].update(
    model_path=str(MODEL_PATH),
    local_files_only=True,
)
cfg["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
cfg["experiment_root"] = str(DRIVE_ROOT)

import copy
from dlm_project.tasks.gsm8k.benchmark import resolve_settings

# Test Profile
PROFILE = "smoke"
NUM_SAMPLES = 2  # Number of questions to evaluate

TEMPERATURE = 1.0
SEED = 1
GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 128

cfg["data"]["num_samples"] = NUM_SAMPLES
cfg["evaluation"]["profile"] = PROFILE
cfg["generation"].update(gen_length=GEN_LENGTH, steps=STEPS,
                          block_length=BLOCK_LENGTH, temperature=TEMPERATURE)
cfg["decoding"]["seed"] = SEED
# Validate without replacing the nullable sample count with a profile-specific count.
resolved = resolve_settings("confidence", cfg, expected_family="llada")
BENCH_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(BENCH_CONFIG.read_text())
print("Selected questions:", resolved["data"]["num_samples"])

BENCH_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))

model:
  family: llada
  model_path: /content/models/LLaDA-8B-Instruct
  device: cuda
  dtype: bfloat16
  local_files_only: true
data:
  num_samples: 2
  cache_dir: /content/drive/MyDrive/CSCI544-DLM/cache/datasets
  revision: null
prompt:
  instructions: 'Solve this math problem in English.

    Use at most three short calculation lines.

    End with a final line containing exactly #### followed by a plain number.

    Do not include units, %, LaTeX, or any text after that number.'
generation:
  gen_length: 256
  steps: 256
  block_length: 128
  temperature: 1.0
decoding:
  seed: 1
evaluation:
  profile: smoke
experiment_root: /content/drive/MyDrive/CSCI544-DLM
experiment_timezone: America/Los_Angeles

Selected questions: 2


713

## Decoding Method
Methods


*   Random Decoding
*   top_k_margin
*   best_of_5
*   confidence
*   eb_sampler


### Confidence Based Decoding

In [9]:
import subprocess
import sys

METHOD = "confidence"
# Other choices: "random_mdm", "top_k_margin", "best_of_5"

PROFILE = "smoke"
# smoke: 20 questions; initial: 100; full: 1,319

subprocess.run(
    [
        sys.executable, "-u",
        str(REPO_DIR / "scripts" / f"eval_gsm8k_{METHOD}.py"),
        "--config", str(BENCH_CONFIG),
        "--profile", PROFILE,
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/CSCI544-DLM/scripts/eval_gsm8k_confidence.py', '--config', '/content/gsm8k-decoding.yaml', '--profile', 'smoke'], returncode=0)

In [10]:
import json
import random

folder = DRIVE_ROOT / "LLaDA" / "GSM8K" / METHOD / PROFILE
files = list(folder.rglob("predictions.jsonl"))
assert files, f"No saved predictions found in {folder}"

latest = max(files, key=lambda p: p.stat().st_mtime)
records = [
    json.loads(line)
    for line in latest.read_text().splitlines()
    if line.strip()
]

print(f"Showing predictions from: {latest.parent}\n")

for row in random.sample(records, min(5, len(records))):
    question = row["prompt"].split("\n\nQuestion:\n", 1)[-1]

    print("=" * 80)
    print(f"Question ({row['example_id']}):\n{question}")
    print(f"\nModel response:\n{row['raw_output']}")
    print(f"\nExtracted answer: {row['predicted_answer']}")
    print(f"Correct answer:   {row['reference_answer']}")
    print(f"Correct:          {row['exact_match']}\n")

Showing predictions from: /content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K/confidence/smoke/gen256-steps256-blk128-temp1-seed1/11:10:26:16:39:01

Question (test:0):
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Model response:
#### 26

Extracted answer: 26
Correct answer:   18
Correct:          False

Question (test:1):
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

Model response:
#### 3

Extracted answer: 3
Correct answer:   3
Correct:          True



### Random Decoding

In [11]:
import subprocess
import sys

METHOD = "random_mdm"
# Other choices: "random_mdm", "top_k_margin", "best_of_5"

PROFILE = "smoke"
# smoke: 20 questions; initial: 100; full: 1,319

subprocess.run(
    [
        sys.executable, "-u",
        str(REPO_DIR / "scripts" / f"eval_gsm8k_{METHOD}.py"),
        "--config", str(BENCH_CONFIG),
        "--profile", PROFILE,
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/CSCI544-DLM/scripts/eval_gsm8k_random_mdm.py', '--config', '/content/gsm8k-decoding.yaml', '--profile', 'smoke'], returncode=0)

In [12]:
import json
import random

folder = DRIVE_ROOT / "LLaDA" / "GSM8K" / METHOD / PROFILE
files = list(folder.rglob("predictions.jsonl"))
assert files, f"No saved predictions found in {folder}"

latest = max(files, key=lambda p: p.stat().st_mtime)
records = [
    json.loads(line)
    for line in latest.read_text().splitlines()
    if line.strip()
]

print(f"Showing predictions from: {latest.parent}\n")

for row in random.sample(records, min(5, len(records))):
    question = row["prompt"].split("\n\nQuestion:\n", 1)[-1]

    print("=" * 80)
    print(f"Question ({row['example_id']}):\n{question}")
    print(f"\nModel response:\n{row['raw_output']}")
    print(f"\nExtracted answer: {row['predicted_answer']}")
    print(f"Correct answer:   {row['reference_answer']}")
    print(f"Correct:          {row['exact_match']}\n")

Showing predictions from: /content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K/random_mdm/smoke/gen256-steps256-blk128-temp1-seed1/11:10:26:16:41:19

Question (test:0):
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Model response:
Start with 16 eggs per day. subtract 3 eggs for breakfast and 4 for muffins: 16 - 3 = 4 eggs left. subtracting the muffins: fewer remaining eggs: 4 - eggs sold to friends = 13 eggs sold. 13 sold eggs at $2 each: 13 eggs * dollars per egg = 26 dollars daily. ####26

Extracted answer: None
Correct answer:   18
Correct:          False

Question (test:1):
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

Model response:
First, display 2 bolts for blue fiber.
Divide 2 by 2 to get 1 

### Top-K Margin Decoding

In [17]:
import subprocess
import sys

METHOD = "top_k_margin"
# Other choices: "random_mdm", "top_k_margin", "best_of_5"

PROFILE = "smoke"
# smoke: 20 questions; initial: 100; full: 1,319

subprocess.run(
    [
        sys.executable, "-u",
        str(REPO_DIR / "scripts" / f"eval_gsm8k_{METHOD}.py"),
        "--config", str(BENCH_CONFIG),
        "--profile", PROFILE,
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/CSCI544-DLM/scripts/eval_gsm8k_top_k_margin.py', '--config', '/content/gsm8k-decoding.yaml', '--profile', 'smoke'], returncode=0)

In [14]:
import json
import random

folder = DRIVE_ROOT / "LLaDA" / "GSM8K" / METHOD / PROFILE
files = list(folder.rglob("predictions.jsonl"))
assert files, f"No saved predictions found in {folder}"

latest = max(files, key=lambda p: p.stat().st_mtime)
records = [
    json.loads(line)
    for line in latest.read_text().splitlines()
    if line.strip()
]

print(f"Showing predictions from: {latest.parent}\n")

for row in random.sample(records, min(5, len(records))):
    question = row["prompt"].split("\n\nQuestion:\n", 1)[-1]

    print("=" * 80)
    print(f"Question ({row['example_id']}):\n{question}")
    print(f"\nModel response:\n{row['raw_output']}")
    print(f"\nExtracted answer: {row['predicted_answer']}")
    print(f"Correct answer:   {row['reference_answer']}")
    print(f"Correct:          {row['exact_match']}\n")

Showing predictions from: /content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K/top_k_margin/smoke/gen256-steps256-blk128-temp1-seed1/11:10:26:16:43:09

Question (test:1):
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

Model response:
- Total blue fiber: 2
- Total white fiber: half of blue (2/2) = 1

#### 3

Extracted answer: 3
Correct answer:   3
Correct:          True

Question (test:0):
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Model response:
Janet has 16 eggs - 3 eggs =13 eggs after eating for breakfast.
She has 13 eggs - 4 eggs = 9 eggs left after baking muffins.
Since she sells each egg for $2, Janet makes 9 eggs x $2/egg at the farmers' market.
#### 22

Extracted answer: 22
Correct answer: 

### Best of 5 Decoding


In [15]:
import subprocess
import sys

METHOD = "best_of_5"
# Other choices: "random_mdm", "top_k_margin", "best_of_5"

PROFILE = "smoke"
# smoke: 20 questions; initial: 100; full: 1,319

subprocess.run(
    [
        sys.executable, "-u",
        str(REPO_DIR / "scripts" / f"eval_gsm8k_{METHOD}.py"),
        "--config", str(BENCH_CONFIG),
        "--profile", PROFILE,
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/CSCI544-DLM/scripts/eval_gsm8k_best_of_5.py', '--config', '/content/gsm8k-decoding.yaml', '--profile', 'smoke'], returncode=0)

In [16]:
import json
import random

folder = DRIVE_ROOT / "LLaDA" / "GSM8K" / METHOD / PROFILE
files = list(folder.rglob("predictions.jsonl"))
assert files, f"No saved predictions found in {folder}"

latest = max(files, key=lambda p: p.stat().st_mtime)
records = [
    json.loads(line)
    for line in latest.read_text().splitlines()
    if line.strip()
]

print(f"Showing predictions from: {latest.parent}\n")

for row in random.sample(records, min(5, len(records))):
    question = row["prompt"].split("\n\nQuestion:\n", 1)[-1]

    print("=" * 80)
    print(f"Question ({row['example_id']}):\n{question}")
    print(f"\nModel response:\n{row['raw_output']}")
    print(f"\nExtracted answer: {row['predicted_answer']}")
    print(f"Correct answer:   {row['reference_answer']}")
    print(f"Correct:          {row['exact_match']}\n")

Showing predictions from: /content/drive/MyDrive/CSCI544-DLM/LLaDA/GSM8K/best_of_5/smoke/gen256-steps256-blk128-temp1-seed1/11:10:26:16:44:57

Question (test:1):
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

Model response:
#### 3

Extracted answer: 3
Correct answer:   3
Correct:          True

Question (test:0):
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Model response:
#### 26

Extracted answer: 26
Correct answer:   18
Correct:          False



### EB-Sampler (Entropy-Bounded Sampling)

Global EB selects the largest prefix of positions ranked by error whose entropy sum minus maximum is at most `gamma`, following [Section 4, Figure 4, and Algorithm 1](https://arxiv.org/html/2505.24857). Entropy and rankings use **unscaled probabilities** with the mask token excluded. Temperature controls token sampling only and defaults to **0** (greedy tokens). Default `gamma=0.1` nats is configurable; use `0` for conservative selection or `float("inf")` to fill all remaining targets in one forward. Error proxies: `confidence`, `entropy`, and `margin`.

This section inherits the cached model, prompt, seed, question count, and frozen test manifest. Selection is global, and `steps` does not cap EB's adaptive completion. Saved results include actual NFEs, iterations, tokens committed per iteration, and their average. MCTS and its comparison remain opt-in below.


In [ ]:
import copy
from pathlib import Path
import yaml
from dlm_project.tasks.gsm8k.benchmark import resolve_settings

EB_GAMMA = 0.1
EB_ERROR_PROXY = "confidence"
EB_TEMPERATURE = 0.0

eb_cfg = copy.deepcopy(cfg)
eb_cfg["generation"].update(steps=eb_cfg["generation"]["gen_length"],
                            block_length=eb_cfg["generation"]["gen_length"],
                            temperature=EB_TEMPERATURE)
eb_cfg["decoding"] = {
    "method": "eb_sampler",
    "seed": cfg["decoding"]["seed"],
    "eb_sampler": {"gamma": EB_GAMMA, "error_proxy": EB_ERROR_PROXY},
}
eb_cfg["experiment_timezone"] = EXPERIMENT_TIMEZONE
eb_resolved = resolve_settings("eb_sampler", eb_cfg, expected_family="llada")
# Keep nullable profile counts, but serialize infinity as a JSON/YAML-safe string.
eb_cfg["decoding"]["eb_sampler"] = eb_resolved["decoding"]["eb_sampler"]
EB_CONFIG = Path("/content/gsm8k-eb-sampler.yaml")
EB_MANIFEST = DRIVE_ROOT / "manifests/gsm8k_test.json"
EB_CONFIG.write_text(yaml.safe_dump(eb_cfg, sort_keys=False))
EB_RUN_DIR = None

print(EB_CONFIG.read_text())
print("Selected questions:", eb_resolved["data"]["num_samples"])
print("Shared test manifest:", EB_MANIFEST)
print("Entropy and ranking: unscaled; sampling temperature:", EB_TEMPERATURE)


In [ ]:
import json
import subprocess
import sys
import uuid


def run_eb_sampler():
    global EB_RUN_DIR
    EB_RUN_DIR = None
    receipt_path = Path("/content") / f"llada-gsm8k-eb-sampler-{uuid.uuid4().hex}.json"
    command = [
        sys.executable, "-u",
        str(REPO_DIR / "scripts/eval_gsm8k_eb_sampler.py"),
        "--config", str(EB_CONFIG),
        "--profile", eb_cfg["evaluation"]["profile"],
        "--manifest", str(EB_MANIFEST),
        "--run-manifest", str(receipt_path),
    ]
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    receipt = json.loads(receipt_path.read_text())
    if (receipt["status"] != "completed" or receipt["model_family"] != "llada"
            or receipt["method"] != "eb_sampler"
            or receipt["profile"] != eb_cfg["evaluation"]["profile"]):
        raise RuntimeError("The run receipt does not match this EB evaluation.")
    EB_RUN_DIR = Path(receipt["run_dir"])
    print("This run:", EB_RUN_DIR)
    return EB_RUN_DIR


run_eb_sampler()


In [ ]:
import json
import random


def show_eb_answers(count=5):
    if type(count) is not int or count < 1:
        raise ValueError("count must be a positive integer.")
    if EB_RUN_DIR is None:
        raise RuntimeError("Complete the EB evaluation before previewing its answers.")
    metrics = json.loads((EB_RUN_DIR / "metrics.json").read_text())
    records = [json.loads(line)
               for line in (EB_RUN_DIR / "predictions.jsonl").read_text().splitlines()
               if line.strip()]
    print("Saved metrics:")
    print(json.dumps(metrics, indent=2))
    print("Showing saved predictions from:", EB_RUN_DIR)
    selected = random.sample(records, min(count, len(records)))
    for row in selected:
        question = row.get("base_prompt", row["prompt"]).rsplit("\n\nQuestion:\n", 1)[-1]
        print("=" * 80)
        print(f"Question ({row['example_id']}):\n{question}")
        print(f"\nModel response:\n{row['raw_output']}")
        print(f"\nExtracted answer: {row['predicted_answer']}")
        print(f"Correct answer:   {row['reference_answer']}")
        print(f"Correct:          {row['exact_match']}")
        print(f"Gamma: {row['gamma']} | Proxy: {row['error_proxy']} | Temperature: {row['temperature']}")
        print(f"NFEs: {row['generation_nfes']} | Iterations: {row['decoding_steps']}")
        print("Tokens unmasked per iteration:", row["tokens_unmasked_per_iteration"])
        print("Average tokens per iteration:", row["average_tokens_unmasked_per_iteration"])
        print()
    return selected


show_eb_answers()


## MEDAL paper reconstruction: MCTS + adjusted confidence

This optional MCTS method retains the active two-question override and frozen GSM8K test manifest. Setting `NUM_SAMPLES = None` restores smoke (20), initial (100), or full (1,319) questions.

Search uses immediate parent-to-child information gain, cached candidate rewards, and literal adjusted-score softmax final completion. Inline three-subtask prompting uses disjoint training demonstrations, never the evaluated question's reference answer. These are documented reconstruction choices; author-code parity is unverified.

Search has **no default forward cap** and must collect three depth-20 candidates. Search costs one root forward plus one forward per evaluated child; candidate ranking adds no forwards. With 256 output tokens, final completion at depth 20 costs 236 forwards. A direct-path example costs 335 forwards; broader exploration can cost more. Use an explicit finite cap with `stop_mode="budget"` only for a labeled budget-limited ablation.

Disabled by default. Set `RUN_MCTS = True` in the next cell to enable search and its comparison workflow.


In [ ]:
import copy
from pathlib import Path
import yaml
from dlm_project.tasks.gsm8k.benchmark import resolve_settings

RUN_MCTS = False

MCTS_SETTINGS = {
    "init_tokens": 20,
    "num_candidates": 3,
    "top_k_tokens": 3,
    "top_k_actions": 5,
    "max_model_calls": None,
    "margin_gamma": 10.0,
    "exploration_c": 0.01,
    "epsilon": 1e-8,
    "stop_mode": "candidates",
}

# Copy the active benchmark config, preserving its model, prompt, and controls.
mcts_cfg = copy.deepcopy(cfg)
# The MEDAL sampler selects positions globally; the other four configs stay unchanged.
mcts_cfg["generation"].update(steps=mcts_cfg["generation"]["gen_length"],
                              block_length=mcts_cfg["generation"]["gen_length"], temperature=1.0)
mcts_cfg["decoding"] = {
    "method": "mcts_confidence",
    "seed": cfg["decoding"]["seed"],
    "mcts": MCTS_SETTINGS,
}
mcts_cfg["experiment_timezone"] = EXPERIMENT_TIMEZONE
mcts_resolved = resolve_settings("mcts_confidence", mcts_cfg, expected_family="llada")
MCTS_CONFIG = Path("/content/gsm8k-mcts-confidence.yaml")
MCTS_MANIFEST = DRIVE_ROOT / "manifests/gsm8k_test.json"
MCTS_CONFIG.write_text(yaml.safe_dump(mcts_cfg, sort_keys=False))
MCTS_RUN_DIR = None

print(MCTS_CONFIG.read_text())
print("Selected questions:", mcts_resolved["data"]["num_samples"])
print("Shared test manifest:", MCTS_MANIFEST)


In [ ]:
import json
import subprocess
import sys
import uuid


def run_mcts_confidence():
    global MCTS_RUN_DIR
    MCTS_RUN_DIR = None
    receipt_path = Path("/content") / f"llada-gsm8k-mcts-{uuid.uuid4().hex}.json"
    command = [
        sys.executable, "-u",
        str(REPO_DIR / "scripts/eval_gsm8k_mcts_confidence.py"),
        "--config", str(MCTS_CONFIG),
        "--profile", mcts_cfg["evaluation"]["profile"],
        "--manifest", str(MCTS_MANIFEST),
        "--run-manifest", str(receipt_path),
    ]
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    receipt = json.loads(receipt_path.read_text())
    if (receipt["status"] != "completed" or receipt["model_family"] != "llada"
            or receipt["method"] != "mcts_confidence"
            or receipt["profile"] != mcts_cfg["evaluation"]["profile"]):
        raise RuntimeError("The run receipt does not match this MCTS evaluation.")
    MCTS_RUN_DIR = Path(receipt["run_dir"])
    print("This run:", MCTS_RUN_DIR)
    return MCTS_RUN_DIR


if RUN_MCTS:
    run_mcts_confidence()
else:
    print("MCTS is disabled. Set RUN_MCTS = True to run it.")


In [ ]:
import json
import random


def show_mcts_answers(count=5):
    if type(count) is not int or count < 1:
        raise ValueError("count must be a positive integer.")
    if MCTS_RUN_DIR is None:
        raise RuntimeError("Complete the MCTS evaluation before previewing its answers.")
    metrics = json.loads((MCTS_RUN_DIR / "metrics.json").read_text())
    records = [json.loads(line)
               for line in (MCTS_RUN_DIR / "predictions.jsonl").read_text().splitlines()
               if line.strip()]
    print("Saved metrics:")
    print(json.dumps(metrics, indent=2))
    print("Showing saved predictions from:", MCTS_RUN_DIR)
    if not records:
        print("No completed predictions are available.")
        return []
    selected = random.sample(records, min(count, len(records)))
    for row in selected:
        question = row.get("base_prompt", row["prompt"]).rsplit("\n\nQuestion:\n", 1)[-1]
        print("=" * 80)
        print(f"Question ({row['example_id']}):\n{question}")
        print(f"\nModel response:\n{row['raw_output']}")
        print(f"\nExtracted answer: {row['predicted_answer']}")
        print(f"Correct answer:   {row['reference_answer']}")
        print(f"Correct:          {row['exact_match']}")
        print(f"Initialization depth: {row['selected_depth']}/{row['requested_depth']}")
        print(f"Candidates: {row['candidate_count']} | Stop: {row['stop_reason']}")
        print(f"Fallback: {row['fallback_used']} | Reason: {row['fallback_reason']}")
        print(f"Forwards: search={row['search_model_calls']}, "
              f"completion={row['completion_model_calls']}, total={row['total_model_calls']}")
        print("Implementation:", row.get("implementation_version", "historical"))
        print("Forward phases:", {phase: row.get(f"{phase}_model_calls", 0)
                                  for phase in ("tree_analysis", "simulation", "candidate_evaluation")})
        print("Phase seconds:", {phase: row.get(f"{phase}_latency_seconds", 0)
                                  for phase in ("tree_analysis", "simulation", "candidate_evaluation", "completion")})
        print("Candidate rewards:", [(candidate["node_id"], candidate["reward"])
                                      for candidate in row.get("candidate_evaluations", [])])
        print()
    return selected


if RUN_MCTS and MCTS_RUN_DIR is not None:
    show_mcts_answers()


### MEDAL artifacts and larger evaluations

The existing method directory now includes a `medal-v2` settings label, so historical full-rollout runs remain distinguishable. Unique run receipts select the exact invocation. Metrics include root analysis, every immediate-child evaluation, and final completion; failure preserves completed predictions and streamed partial traces.

Rerun previews to read saved answers without inference. Change the shared `NUM_SAMPLES` to `None` for profile counts. Keep temperature 1.0, global block length equal to output length, and `max_model_calls=None` for the uncapped candidate workflow. Gamma 10 and exploration constant 0.01 are repository defaults rather than verified author settings.


## Accuracy, compute, and MCTS simulation comparison

After the setup and benchmark configuration cells, you can skip the individual method runs above and run this section directly.

This optional experiment loads **one LLaDA model in one subprocess** and runs the four baselines plus MCTS with simulation caps of 4, 8, 16, 32, and 64. It uses the same frozen questions, per-question seeds, temperature, 256 output slots, single global block, and MEDAL two-shot prompt for every condition. These comparison runs have their own artifacts; the individual runs above retain their behavior.

A **simulation** is one immediate-child evaluation and backup. Each costs one forward, with one additional root-analysis forward per search. Candidate ranking uses cached rewards and adds no forwards. A low simulation cap can stop before depth 20; inspect the saved selected depth and fallback rate. Start with two questions to check runtime and artifacts. Change the count to 100 only after a smoke check.

The table reports strict accuracy with Wilson 95% intervals, total NFEs, mean wall-clock time, decoding steps, and completed simulations. Three plots compare accuracy against simulations, NFEs, and time. Best-of-5 decoding steps include all five candidates; MCTS decoding steps cover final completion, with internal simulation steps saved separately. All runs record batch size 1, average evaluated search depth, unique evaluated child states, selected/max depth, and phase costs. Entropy (nats) and top-two margin use existing predictions, so signal collection adds zero NFEs but its processing time is included in latency.
Set `RUN_MCTS = True` above to launch this optional MCTS comparison. Saved-run reports can still be opened without enabling search.


In [ ]:
import copy
from dataclasses import asdict
import json
from pathlib import Path
import yaml
from dlm_project.tasks.gsm8k.benchmark import resolve_settings
from dlm_project.decoding.mcts_decoding import MCTSConfig

METRICS_NUM_SAMPLES = 2
METRICS_SIMULATION_BUDGETS = [4, 8, 16, 32, 64]
METRICS_PROFILE = cfg["evaluation"]["profile"]
metrics_cfg = copy.deepcopy(cfg)
metrics_cfg["data"]["num_samples"] = METRICS_NUM_SAMPLES
metrics_cfg["evaluation"]["profile"] = METRICS_PROFILE
metrics_cfg["generation"].update(steps=metrics_cfg["generation"]["gen_length"],
                                 block_length=metrics_cfg["generation"]["gen_length"])
metrics_cfg["prompt"]["medal_decomposition"] = True
METRICS_MCTS_SETTINGS = asdict(MCTSConfig())
metrics_cfg["decoding"]["mcts"] = METRICS_MCTS_SETTINGS
metrics_cfg["experiment_timezone"] = EXPERIMENT_TIMEZONE
METRICS_CONFIG = Path("/content/gsm8k-primary-metrics.yaml")
METRICS_MANIFEST = DRIVE_ROOT / "manifests/gsm8k_test.json"
resolve_settings("confidence", metrics_cfg, expected_family="llada")
METRICS_CONFIG.write_text(yaml.safe_dump(metrics_cfg, sort_keys=False))
METRICS_COMPARISON_DIR = None
# Alternatively, paste explicit historical run directories here and run only the report cell.
METRICS_RUN_DIRS = {}
print(METRICS_CONFIG.read_text())
print("Simulation caps:", METRICS_SIMULATION_BUDGETS)
print("Each cap limits immediate-child evaluations; total NFEs also include root analysis and final completion.")


In [ ]:
import subprocess
import sys
import uuid


def run_primary_metrics():
    global METRICS_COMPARISON_DIR
    METRICS_COMPARISON_DIR = None
    receipt_path = Path("/content") / f"llada-gsm8k-metrics-{uuid.uuid4().hex}.json"
    command = [sys.executable, "-u", str(REPO_DIR / "scripts/run_gsm8k_metrics.py"),
               "--config", str(METRICS_CONFIG), "--profile", METRICS_PROFILE,
               "--num-samples", str(METRICS_NUM_SAMPLES),
               "--manifest", str(METRICS_MANIFEST),
               "--run-manifest", str(receipt_path),
               "--budgets", *map(str, METRICS_SIMULATION_BUDGETS)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    receipt = json.loads(receipt_path.read_text())
    if (receipt["status"] != "completed" or receipt["model_family"] != "llada"
            or receipt["task"] != "gsm8k_metrics"):
        raise RuntimeError("The receipt does not match this comparison.")
    METRICS_COMPARISON_DIR = Path(receipt["run_dir"])
    print("This comparison:", METRICS_COMPARISON_DIR)
    return METRICS_COMPARISON_DIR


if RUN_MCTS:
    run_primary_metrics()
else:
    print("The MCTS comparison is disabled (RUN_MCTS = False).")


In [ ]:
if METRICS_COMPARISON_DIR is not None or METRICS_RUN_DIRS:
    from IPython.display import display, Markdown, Image, FileLink
    from dlm_project.experiments.primary_metrics import generate_primary_report

    if METRICS_COMPARISON_DIR is not None:
        METRICS_RUN_DIRS = json.loads((METRICS_COMPARISON_DIR / "runs.json").read_text())
        report_dir = METRICS_COMPARISON_DIR / "report"
        expected_conditions = ["random_mdm", "top_k_margin", "confidence", "best_of_5",
                               *(f"mcts_{budget}_simulations" for budget in METRICS_SIMULATION_BUDGETS)]
    else:
        if not METRICS_RUN_DIRS:
            raise RuntimeError("Run the comparison or select explicit saved directories in METRICS_RUN_DIRS.")
        report_dir = DRIVE_ROOT / "LLaDA/GSM8K/selected-metrics-report"
        expected_conditions = list(METRICS_RUN_DIRS)

    # Offline: no weights are loaded and no dataset is downloaded.
    primary_summary = generate_primary_report(METRICS_RUN_DIRS, report_dir,
                                             expected_conditions=expected_conditions)
    display(Markdown((report_dir / "report.md").read_text().split("![", 1)[0]))
    for stem in ("accuracy_vs_simulations", "accuracy_vs_nfes", "accuracy_vs_latency"):
        display(Image(filename=str(report_dir / "plots" / f"{stem}.png")))
        display(FileLink(str(report_dir / "plots" / f"{stem}.pdf")))
    for filename in ("summary.csv", "results.csv", "plot_data.json", "report.md"):
        display(FileLink(str(report_dir / filename)))

    if METRICS_COMPARISON_DIR is not None:
        print("Setup costs (excluded from per-question NFEs and time):")
        print((METRICS_COMPARISON_DIR / "setup.json").read_text())
else:
    print("No comparison selected; skipping its report.")


In [ ]:
if METRICS_RUN_DIRS:
    # Inspect the saved uncertainty signals without another model forward.
    SIGNAL_CONDITION = "confidence"
    SIGNAL_EXAMPLE_INDEX = 0
    signal_run = Path(METRICS_RUN_DIRS[SIGNAL_CONDITION])
    signal_records = [json.loads(line) for line in (signal_run / "predictions.jsonl").read_text().splitlines()
                      if line.strip()]
    if not signal_records:
        raise RuntimeError("No completed predictions in the selected condition.")
    signal_row = signal_records[SIGNAL_EXAMPLE_INDEX]
    signal_events = [json.loads(line) for line in (signal_run / signal_row["trace_file"]).read_text().splitlines()
                     if line.strip()]
    fields = ("stage", "phase", "candidate_index", "step", "signal_nfe", "masked_count",
              "mean_token_entropy", "mean_confidence_margin", "signal_extra_nfes")
    print("Condition:", SIGNAL_CONDITION, "| Example:", signal_row["example_id"])
    for event in [event for event in signal_events if "mean_token_entropy" in event][:10]:
        print({key: event.get(key) for key in fields})
    print("All states are retained in:", signal_run / signal_row["trace_file"])
else:
    print("No comparison selected; skipping its signal preview.")
